# Importing libraries

In [ ]:
# Necessary Libraries
import numpy as np # (for numeric operations)
import pandas as pd # (for data manipulation)
import matplotlib.pyplot as plt # (for plotting)
import seaborn as sns # (for advanced plotting)
import sklearn as skl # (machine learning library)
from datetime import date # (for date operations)

# Week 0

## Loading the data and early inspection

In [ ]:
df_players = pd.read_csv('data/raw/dataset/players.csv')

for i in df_players.columns:
    print(i)

There are 23 columns present in this table

In [ ]:
# Data Cleaning
# Drop columns that are not useful for analysis
df_players = df_players.drop(columns = ['url', 'image_url', 'city_of_birth', 'country_of_birth', 'contract_expiration_date', 'first_name', 'last_name'])

df_players.head()

In [ ]:
# Convert date_of_birth to datetime and finding age of players
df_players['date_of_birth'] = pd.to_datetime(df_players['date_of_birth'])

def calculate_age(born):
  today = date.today()
  return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

df_players['age'] = df_players['date_of_birth'].apply(calculate_age)

df_players.head()

## Week 1

In [ ]:
# 1) Basic inspection
# Load the data into pandas
df_players.info()

In [ ]:
df_players.describe()

In [ ]:
# Create figure with 2 subplots
fig, axes = plt.subplots(2, 1, figsize=(10, 8), gridspec_kw={'height_ratios': [3, 1]})

# --- Histogram (with log scale) ---
sns.histplot(df_players['market_value_in_eur'], bins=50, ax=axes[0])
axes[0].set_xscale("log")
axes[0].set_title("Market Value Distribution (log scale)")
axes[0].set_xlabel("Market Value (EUR)")
axes[0].set_ylabel("Count")

# --- Boxplot (log transformed values) ---
sns.boxplot(x=np.log1p(df_players['market_value_in_eur']), ax=axes[1], orient="h")
axes[1].set_title("Market Value Spread (log scaled)")
axes[1].set_xlabel("log(Market Value + 1)")

plt.tight_layout()
plt.show()

In [ ]:
# 2) Data Quality Checks
# a) Missing Values
df_players.isnull().sum()

In [ ]:
# b) Duplicate row check
df_players.duplicated().sum()

In [ ]:
# 3)  Demographic analysis
# a) Age distribution

plt.figure(figsize = (10, 6))
sns.histplot(df_players['age'], bins = 30, kde = True)
plt.title('Age Distribution of Players')
plt.xlabel('Age')
plt.ylabel('Number of Players')
plt.tight_layout()
plt.show()

In [ ]:
# b) Nationality: Top 10 Most Common Nationalities

top_10_player_countries = df_players['country_of_citizenship'].value_counts().head(10)

plt.bar(top_10_player_countries.index, top_10_player_countries.values)
plt.xticks(rotation = 45)
plt.title('Top 10 Player Countries')
plt.xlabel('Country')
plt.ylabel('Number of Players')
plt.tight_layout()
plt.show()

In [ ]:
# c) Position Counts
df_players['position'].value_counts().plot.bar()
plt.xticks(rotation = 45)
plt.tight_layout()
plt.show()

In [ ]:
# d) Position by country
df_players.groupby('country_of_citizenship')['position'].value_counts().unstack().fillna(0)

#.sort_values(by = 'position', ascending = False).head(10)

In [ ]:
# Create pivot table with counts per position
count_df = df_players.groupby('country_of_citizenship')['position'].value_counts().unstack().fillna(0)
# Create a new column 'Total' for total player count per country
count_df['Total'] = count_df.sum(axis=1)
# Sort based on the 'Total' count in descending order and show top 10 countries
sorted_count_df = count_df.sort_values(by='Total', ascending=False).head(10)
sorted_count_df

In [ ]:
# 4) Physical Attributes
# a) Height Distribution

sns.histplot(df_players['height_in_cm'], bins=30, kde=True)
plt.title('Height Distribution of Players')
plt.xlabel('Height (cm)')
plt.ylabel('Number of Players')
plt.tight_layout()
plt.show()

In [ ]:
# 5) Career Information
# a) Number of unique clubs represented in the dataset
df_players['current_club_id'].nunique()

In [ ]:
# b) Count how many players are free agents (not currently signed to any club)
df_players['current_club_id'].isnull().value_counts()

There are literlly no free agents as shown in the above code!!

In [ ]:
# Top 10 clubs by player count
club_age_stats = (
    df_players.groupby("current_club_name")["age"]
    .agg(["count", "mean", "median"])
    .sort_values("count", ascending=False)
)

print(club_age_stats.head(10))

In [ ]:
df_players['date_of_birth'] = pd.to_datetime(df_players['date_of_birth'])
df_players['age'] = (pd.Timestamp.today() - df_players['date_of_birth']).dt.days // 365

In [ ]:
# Age distribution
df_players['age'].value_counts().sort_index().plot(kind='bar', 
       figsize=(12,6), 
       title='Player Age Distribution')
plt.ylabel('Occurances')
plt.show()

In [ ]:
df_players["age"].plot(kind="hist", 
                       bins=30, 
                       figsize=(8,6), 
                       title="Age Distribution of Players")
plt.show()

In [ ]:
# c) Age Vs Current club
top20_clubs = (
    df_players['current_club_id']
    .value_counts()
    .head(20)
    .index
)

sample = df_players[df_players['current_club_id'].isin(top20_clubs)]

plt.figure(figsize=(12,6))
plt.scatter(sample['current_club_id'], sample['age'], alpha=0.5)
plt.xticks(rotation=90)
plt.title("Player Ages by Club (Top 20 Clubs)")
plt.xlabel("Current Club")
plt.ylabel("Age")
plt.show()

In [ ]:
top_clubs = df_players['current_club_id'].value_counts().head(20).index
df_top = df_players[df_players['current_club_id'].isin(top_clubs)]

plt.figure(figsize=(12,6))
sns.boxplot(x="current_club_id", y="age", data=df_top)
plt.title("Player Age Distribution by Club (Top 20 Clubs)")
plt.xlabel("Club ID")
plt.ylabel("Age")
plt.xticks(rotation=45)
plt.show()

In [ ]:
df_players.info()

In [ ]:
# Assuming season or year reference, e.g., 2021-01-01
reference_date = pd.to_datetime("2021-01-01")  

df_players['age'] = (reference_date - df_players['date_of_birth']).dt.days // 365

In [ ]:
df_players['age'].describe()

In [ ]:
df_players = df_players[(df_players['age'] >= 15) & (df_players['age'] <= 40)]

In [ ]:
df_players.head().sort_values(by = 'age', ascending = False)

In [ ]:
df_players['age'] = df_players['age'].astype(int)

df_players.info()

In [ ]:
df_players.loc[df_players['current_club_name'] == "Paris Saint-Germain Football Club", ["name", "age", "market_value_in_eur"]].sort_values(by = 'market_value_in_eur', ascending = False)